### Imports

In [ ]:
from sortedcontainers import SortedList
from fractions import Fraction
from functools import total_ordering

import numpy as np
import pylab as pl
from matplotlib import collections as mc

### Class Segment

In [ ]:
class Segment:
    def __init__(self, x1, y1, x2, y2, currentYList=None):
        self.x1 = x1
        self.x2 = x2
        self.y1 = y1
        self.y2 = y2
        self.currentYList = currentYList
        
        if x1 == x2:
            self.slope = float('inf')
            self.intercept = x1
        else:
            self.slope = (y2 - y1) / (x2 - x1)
            self.intercept = y2 - self.slope * x2

    def currentX(self):
        if self.slope == float('inf'):
            return self.x1
        return (self.currentYList[0] - self.intercept) / self.slope

    def __lt__(self, other):
        return self.currentX() < other.currentX()
    
    def __eq__(self, other):
        return self.currentX() == other.currentX()
        
    def intersection(self, other):
        x1 = self.x1
        y1 = self.y1
        x2 = self.x2
        y2 = self.y2
        x = other.x1
        y = other.y1
        xB = other.x2
        yB = other.y2
        
        dx1 = x2 - x1
        dy1 = y2 - y1
        dx = xB - x
        dy = yB - y
        DET = (-dx1 * dy + dy1 * dx)
        
        if DET == 0:
            raise Exception('Intersection implementation not sufficiently robust for this input.')
        DETinv = Fraction(1, DET)
                                     
        r = Fraction((-dy * (x - x1) + dx * (y - y1)), DET)
        s = Fraction((-dy1 * (x - x1) + dx1 * (y - y1)), DET)

        if r < 0 or r > 1 or s < 0 or s > 1:
            return None
        
        xi = x1 + r * dx1
        yi = y1 + r * dy1
        return (xi, yi)

## Class Event

In [ ]:
class Event:
    class Type:
        INTERSECTION = 0
        START = 1
        END = 2

    def __init__(self, type, segment, segment2=None, ipoint=None):
        self.type = type
        self.segment = segment
        self.segment2 = segment2
        
        if type == 1:
            if segment.y1 < segment.y2 or (segment.y1 == segment.y2 and segment.x2 < segment.x1):
                self.key = (segment.y2, -segment.x2)
            else:
                self.key = (segment.y1, -segment.x1)
            
        elif type == 2:
            if segment.y1 > segment.y2 or (segment.y1 == segment.y2 and segment.x2 > segment.x1):
                self.key = (segment.y2, -segment.x2)
            else:
                self.key = (segment.y1, -segment.x1)
                 
        elif type == 0:
            self.key = (ipoint[1], -ipoint[0])

## Check Intersections

In [ ]:
def checkIntersection(pos, pos2, Events, Status, intersections):
    segment = Status[pos]
    segment2 = Status[pos2]
    ipoint = segment.intersection(segment2)
    if ipoint and ipoint[1] < segment.currentYList[0]:
        ievent = Event(0, segment, segment2, ipoint)
        index = Events.bisect_left(ievent)
        if index == len(Events) or not Events[index].key == (ipoint[1], -ipoint[0]):
            Events.add(ievent)
            intersections.append(ipoint)

## Handling Events

In [ ]:
def handleStartEvent(segment, Events, Status, intersections):
    Status.add(segment)
    pos = Status.index(segment)
    if pos > 0:
        checkIntersection(pos - 1, pos, Events, Status, intersections)
    if pos + 1 < len(Status):
        checkIntersection(pos, pos + 1, Events, Status, intersections)

def handleEndEvent(segment, Events, Status, intersections):
    pos = Status.index(segment)
    Status.remove(segment)
    if pos > 0 and pos < len(Status):
        checkIntersection(pos - 1, pos, Events, Status, intersections)

def handleIntersectionEvent(segment, segment2, Events, Status, intersections):
    currentY = segment.currentYList[0]
    segment.currentYList[0] = currentY + 0.00001

    pos = Status.index(segment)
    pos2 = Status.index(segment2)
    Status.remove(segment)
    segment.currentYList[0] = currentY - 0.00001
    Status.add(segment)

    pos_first = min(pos, pos2)

    if pos_first > 0:
        checkIntersection(pos_first - 1, pos_first, Events, Status, intersections)
    if pos_first + 2 < len(Status):
        checkIntersection(pos_first + 1, pos_first + 2, Events, Status, intersections)

    segment.currentYList[0] = currentY

## Reading Segments

In [ ]:
def readSegments(file):
    currentYList = [0]
    segments = []
    with open(file) as f:
        for line in f:
            coord = [int(x) for x in line.split()]
            s = Segment(coord[0], coord[1], coord[2], coord[3], currentYList)
            segments.append(s)
    return segments